In [ ]:
#connecting to google drive
from google.colab import drive
drive.mount('/content/drive')

In [1]:
# 2.Download and Unzip data
!wget https://ceb.nlm.nih.gov/proj/malaria/cell_images.zip
!unzip -q cell_images.zip -d /content/original_data

print("Downloaded the data and unzipped")

'wget' is not recognized as an internal or external command,
operable program or batch file.


Downloaded the data and unzipped


'unzip' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
# 3. Split data and Separate test data
!pip install split-folders
import splitfolders
import shutil

input_folder = '/content/original_data/cell_images'
output_folder = '/content/split_dataset'

print("Data is breaking  (Train 70%, Val 15%, Test 15%)...")
# Divide data
splitfolders.ratio(input_folder, output=output_folder, seed=42, ratio=(0.7, 0.15, 0.15))
print("Data split process finished!")

# save test data in google drive
drive_test_path = '/content/drive/MyDrive/Malaria_Test_Data'
shutil.copytree(output_folder + '/test', drive_test_path, dirs_exist_ok=True)

print(f"Saved the data in your drive. Path: {drive_test_path}")

Data is breaking  (Train 70%, Val 15%, Test 15%)...


ValueError: The provided input folder "/content/original_data/cell_images" does not exists. Your relative path cannot be found from the current working directory "c:\Computer Vision Project\malaria-detection-cv\noteboks".

In [ ]:
import cv2
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator


#  Enhancement (Gaussian Blur )

def apply_blur(img):
    blurred_img = cv2.GaussianBlur(img, (5, 5), 0)
    return blurred_img


#  Normalization 

def apply_normalization(img):
    normalized_img = img / 255.0
    return normalized_img


#Main pipeline that connects above two steps 

def custom_image_pipeline(img):
    img_step1 = apply_blur(img)           
    img_step2 = apply_normalization(img_step1) 


train_datagen = ImageDataGenerator(preprocessing_function=custom_image_pipeline)
val_datagen = ImageDataGenerator(preprocessing_function=custom_image_pipeline)

#Load images and set the parameters
print("Training Data සකස් කරමින් පවතී...")
train_generator = train_datagen.flow_from_directory(
    '/content/split_dataset/train', 
    target_size=(224, 224),         
    batch_size=32,                  
    class_mode='binary'
)

print("\nValidation Data සකස් කරමින් පවතී...")
val_generator = val_datagen.flow_from_directory(
    '/content/split_dataset/val',   
    target_size=(224, 224),         
    batch_size=32,
    class_mode='binary',
    shuffle=False
)